# 🎯 LocateAnything-3B — Đánh giá độ chính xác trên COCO (Kaggle T4)

Đo **Precision / Recall / F1 @ IoU** + **MAE đếm** trên ảnh COCO có nhãn (person / car / bottle) — điểm số thật cho 3 bài toán đếm.

### ⚙️ Trước khi Run All
- Bật **Settings → Accelerator: GPU T4 x2** và **Internet: On** (thiếu 1 trong 2 là lỗi).
- Kaggle cài sẵn **transformers 5.0.0** nhưng model cần **4.57.1**, và Kaggle **revert lại 5.0.0 mỗi lần restart kernel**. `run_eval.py` **tự cài 4.57.1 rồi tự re-exec tiến trình** (không đụng kernel) — bạn chỉ việc Run All theo thứ tự.


## 1) Tải code mới nhất

In [ ]:
%cd /kaggle/working
!rm -rf VisionOS
!git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git
%cd VisionOS/VisionOS
!git checkout -q claude/rebuild-visionos-codebase-tgg0mf
!git pull -q origin claude/rebuild-visionos-codebase-tgg0mf
!git log --oneline -5


## 2) ✅ Kiểm tra bộ test (KHÔNG cần GPU, ~5s)
Chạy **trước khi tốn GPU** — nếu bước này đỏ thì lỗi nằm ở logic, không phải ở model, khỏi tốn thời gian debug CUDA. Gồm 43 unit test (parse/scenario/counting), 71 test `recognition/` (evaluation, geometry, router...) và 8 test mới chạy trên **ảnh COCO thật** (không phải ảnh giả `np.zeros`) cho person/car/bottle.

In [ ]:
!pip install -q -r requirements-test.txt
!pip install -q -r recognition/requirements.txt
!pytest tests tests_recognition -q


## 3) Kiểm chứng bộ chấm điểm (KHÔNG cần GPU)

In [ ]:
!python run_eval.py --selftest


## 4) Phân tích dữ liệu đầu vào (KHÔNG cần model)

In [ ]:
!python run_eval.py --analyze-only --classes person car bottle --n 50


## 5) 🚀 Chạy model — LocateAnything trên GPU (5 ảnh 'person')
**Cell này kích hoạt auto-pin.** Lần đầu thấy:
```
⚙️  transformers==5.0.0 KHÔNG khớp → cài 4.57.1 …
🔄 Khởi động lại với transformers==4.57.1 …
```
Bình thường — script tự re-exec, không cần bạn làm gì thêm.

In [ ]:
!python run_eval.py --model locate --classes person --n 5 --save-dir /kaggle/working/viz


## 5b) 🐞 NẾU cell trên báo lỗi — chạy cell debug CPU này
Nếu Cell 5 dừng vì **CUDA device-side assert** (lỗi mù mờ) HOẶC báo sai phiên bản,
chạy cell này để lấy **lỗi rõ ràng**:
- `LA_DEBUG_CPU=1` → chạy CPU: CUDA assert → biến thành `IndexError` rõ ràng.
- Chỉ 1 ảnh → crash nhanh (nếu là lỗi index).

👉 **Copy TOÀN BỘ output cell này gửi lại** (đặc biệt dòng `🔧 transformers==…`
và dòng `...Error:` cuối) để sửa đúng chỗ.

In [ ]:
# Chạy CPU để lỗi hiện rõ (chậm hơn nhưng crash nhanh nếu là lỗi index).
!LA_DEBUG_CPU=1 python run_eval.py --model locate --classes person --n 1


## 6) 🏁 ĐIỂM SỐ THẬT — 3 lớp person / car / bottle
Chỉ chạy khi Cell 5 đã ra kết quả. Lưu ảnh dự đoán (🟢TP 🔴FP 🟡bỏ sót) vào `/kaggle/working/viz`.

In [ ]:
!python run_eval.py --model locate --classes person car bottle --n 30 --save-dir /kaggle/working/viz


## 7) Xem ảnh dự đoán để soi model sai ở đâu

In [ ]:
import glob
from IPython.display import Image, display
for cls in ['person', 'car', 'bottle']:
    paths = sorted(glob.glob(f'/kaggle/working/viz/{cls}/*.jpg'))[:4]
    print(f'=== {cls}: {len(paths)} ảnh mẫu ===')
    for p in paths:
        display(Image(p))


## 8) (Tùy chọn) So sánh với YOLOv8 (đã học sẵn COCO)
YOLO học thẳng COCO nên cao hơn — mốc tham chiếu. LocateAnything là zero-shot open-vocab.

Baseline đã đo sẵn (YOLOv8n, 126 ảnh COCO train2017 thật, xem `tests_recognition/fixtures`):
`person F1=0.707 · car F1=0.327 · bottle F1=0.333` — dùng để so sánh xem LocateAnything hơn/kém YOLO bao nhiêu trên đúng loại ảnh này.

In [ ]:
!python run_eval.py --model yolo --classes person car bottle --n 30


---
### Đọc kết quả
- **P**: báo đúng bao nhiêu %. **R**: tìm được bao nhiêu vật thật. **F1**: tổng hợp. **MAE**: sai số đếm/ảnh.
- Vật càng nhỏ (bottle thường <1% diện tích ảnh) open-vocab càng khó → F1 thấp là bình thường.
- F1 < 0.45 → cân nhắc fine-tune hoặc chỉnh prompt/NMS trước khi đưa vào production.